# 04 — TimeGAN Training

Three-phase training:
1. Autoencoder pretraining (Embedder + Recovery) — 200 epochs
2. Supervisor pretraining — 200 epochs
3. Joint adversarial training — 500 epochs

**Input:** `data/processed/train_sequences.npy` (GAN sees training data only)  
**Output:** Trained TimeGAN model in `checkpoints/timegan/`

In [ ]:
import sys, os
import numpy as np
import torch
sys.path.insert(0, os.path.abspath('..'))

from src.models.timegan import TimeGAN
from src.training.train_gan import train_timegan, generate_synthetic_sequences

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
CHECKPOINT_DIR = os.path.join('..', 'checkpoints', 'timegan')

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train_sequences = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
print(f'Training sequences: {train_sequences.shape}')
print(f'  N={train_sequences.shape[0]}, seq_len={train_sequences.shape[1]}, features={train_sequences.shape[2]}')

## Train TimeGAN

This takes a while. Adjust epochs for quick iteration vs full training.

In [ ]:
# Full training config (as specified)
model, history = train_timegan(
    train_sequences,
    input_dim=20,
    hidden_dim=64,
    batch_size=128,
    phase1_epochs=200,
    phase2_epochs=200,
    phase3_epochs=500,
    lr=1e-3,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    log_every=20,
)

## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['phase1_loss'])
axes[0].set_title('Phase 1: Autoencoder Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')

axes[1].plot(history['phase2_loss'])
axes[1].set_title('Phase 2: Supervisor Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')

axes[2].plot(history['phase3_d_loss'], label='Discriminator')
axes[2].plot(history['phase3_g_loss'], label='Generator')
axes[2].set_title('Phase 3: Adversarial Losses')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'gan_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## Generate Synthetic Sequences

Generate 50,000 synthetic SPY market sequences and save to disk.

In [ ]:
# If loading a pre-trained model instead of training above:
# model = TimeGAN(input_dim=20, hidden_dim=64)
# model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'timegan_final.pt')))

fake_sequences = generate_synthetic_sequences(
    model,
    n_samples=50000,
    seq_len=60,
    batch_size=1000,
    device=device,
    save_path=os.path.join(PROCESSED_DIR, 'gan_sequences.npy'),
)
print(f'Generated synthetic sequences: {fake_sequences.shape}')